In [2]:
import numpy as np

embeddings = np.load("../models/document_embeddings.npy")

In [3]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=20,
    covariance_type="full",
    random_state=42
)

gmm.fit(embeddings)

,"n_components n_components: int, default=1The number of mixture components.",20
,"covariance_type covariance_type: {'full', 'tied', 'diag', 'spherical'}, default='full'String describing the type of covariance parameters to use.Must be one of:- 'full': each component has its own general covariance matrix.- 'tied': all components share the same general covariance matrix.- 'diag': each component has its own diagonal covariance matrix.- 'spherical': each component has its own single variance.For an example of using `covariance_type`, refer to:ref:`sphx_glr_auto_examples_mixture_plot_gmm_selection.py`.",'full'
,"tol tol: float, default=1e-3The convergence threshold. EM iterations will stop when thelower bound average gain is below this threshold.",0.001
,"reg_covar reg_covar: float, default=1e-6Non-negative regularization added to the diagonal of covariance.Allows to assure that the covariance matrices are all positive.",1e-06
,"max_iter max_iter: int, default=100The number of EM iterations to perform.",100
,"n_init n_init: int, default=1The number of initializations to perform. The best results are kept.",1
,"init_params init_params: {'kmeans', 'k-means++', 'random', 'random_from_data'}, default='kmeans'The method used to initialize the weights, the means and theprecisions.String must be one of:- 'kmeans' : responsibilities are initialized using kmeans.- 'k-means++' : use the k-means++ method to initialize.- 'random' : responsibilities are initialized randomly.- 'random_from_data' : initial means are randomly selected data points... versionchanged:: v1.1 `init_params` now accepts 'random_from_data' and 'k-means++' as initialization methods.",'kmeans'
,"weights_init weights_init: array-like of shape (n_components, ), default=NoneThe user-provided initial weights.If it is None, weights are initialized using the `init_params` method.",None
,"means_init means_init: array-like of shape (n_components, n_features), default=NoneThe user-provided initial means,If it is None, means are initialized using the `init_params` method.",None
,"precisions_init precisions_init: array-like, default=NoneThe user-provided initial precisions (inverse of the covariancematrices).If it is None, precisions are initialized using the 'init_params'method.The shape depends on 'covariance_type':: (n_components,) if 'spherical', (n_features, n_features) if 'tied', (n_components, n_features) if 'diag', (n_components, n_features, n_features) if 'full'",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to the method chosen to initialize theparameters (see `init_params`).In addition, it controls the generation of random samples from thefitted distribution (see the method `sample`).Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42


In [4]:
cluster_probs = gmm.predict_proba(embeddings)
print(cluster_probs.shape)

(19957, 20)


In [5]:
import joblib

joblib.dump(gmm, "../models/gmm_cluster_model.pkl")

print("Cluster model saved")

Cluster model saved


In [6]:
#boundary doc

confidence = cluster_probs.max(axis=1)
print(confidence[:10])

#low confidence docs
boundary_docs = np.argsort(confidence)[:10]
print(boundary_docs)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[ 3428  3459 14135   874  4575 13968  5704 12557  4259  4610]


In [8]:
import pandas as pd

df = pd.read_csv("../data/cleaned_newsgroups.csv")

for idx in boundary_docs:
    print("Document index:", idx)
    print(df["clean_text"][idx][:400])
    print("\n----------------------\n")

Document index: 3428
in article tom huot writes kenneth ng wrote in article tom huot writes peter clark wrote interesting you seem to be the only person i have ever heard of who has had a problem with mag like that i have a mag mx15f myself no problems i liked it so much i showed it to a bunch of my friends 6 of them went out and bought them no problems all gateway 2000 crystal scan monitors are mag innovisions i ve n

----------------------

Document index: 3459
fombaron marc writes is there a more recent version of umbdr522 zip because it doesn t work on my machine my motherboard has symphony sl82c362 chips and they say it will be supported in the later versions so is it out thank you for helping marc the last i heard the author was having some problems in his immediate family and had delayed the continuation of development for a time this was some month

----------------------

Document index: 14135
forwarded from public information office jet propulsion laboratory california instit

Documents with low cluster confidence represent semantic overlap
between topics. For example, discussions about space missions
can overlap between science, politics, and technology clusters.
This demonstrates why fuzzy clustering is appropriate for the
20 Newsgroups dataset.